# Predictive Maintenance — Equipment Failure Prediction (RUL)

**Objective**: Predict the Remaining Useful Life (RUL) of industrial equipment based on sensor measurements from SAP.

**Equipment monitored**:
- BROYEUR_A_BOULETS (Ball Mill) — 8 vibration sensors
- MOTEUR_ENTRAINEMENT (Drive Motor) — 1 temperature + 4 vibration sensors
- MOTEUR_BROYEUR (Mill Motor) — 4 temperature sensors
- TURBOSOUFFLANTE (Turbo Blower) — mixed sensors (temperature, vibration, pressure, displacement)

---

## Step 1 — Data Cleaning & Preparation

### 1.1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully ✓')

Libraries loaded successfully ✓


### 1.2 — Load Raw Data

We use `Documents_mesure_SAP.xlsx` as the source because `DATA.xlsx` has a decimal parsing bug (French comma decimals like `80,37` were read as `8037`).

In [2]:
# Load the raw SAP data
df_raw = pd.read_excel('Documents_mesure_SAP.xlsx')

print(f'Raw data shape: {df_raw.shape}')
print(f'\nColumns: {df_raw.columns.tolist()}')
print(f'\nData types:\n{df_raw.dtypes}')
df_raw.head(10)

Raw data shape: (584, 8)

Columns: ['Poste Technique', 'Point de mesure', 'Date', 'Heure de la mesure', 'ValMes./Val.tot.cpteur', 'Unité caractérist.', 'LimiteInférPlageMesure', 'LimiteSupérPlageMesure']

Data types:
Poste Technique                      str
Point de mesure                    int64
Date                      datetime64[us]
Heure de la mesure                object
ValMes./Val.tot.cpteur               str
Unité caractérist.                   str
LimiteInférPlageMesure               str
LimiteSupérPlageMesure               str
dtype: object


,Poste Technique,Point de mesure,Date,Heure de la mesure,ValMes./Val.tot.cpteur,Unité caractérist.,LimiteInférPlageMesure,LimiteSupérPlageMesure
0,MOTEUR_BROYEUR,87812,2026-06-04,17:20:35,"80,37",°C,NaN,NaN
1,MOTEUR_BROYEUR,87812,2026-06-03,00:00:07,"28,03",°C,NaN,NaN
2,MOTEUR_BROYEUR,87812,2026-06-02,00:00:05,"55,90",°C,NaN,NaN
3,MOTEUR_BROYEUR,87812,2026-06-01,19:05:00,"80,29",°C,NaN,NaN
4,MOTEUR_BROYEUR,87812,2026-05-31,00:00:20,"51,50",°C,NaN,NaN
5,MOTEUR_BROYEUR,87812,2026-05-30,14:28:15,"79,73",°C,NaN,NaN
6,MOTEUR_BROYEUR,87812,2026-05-29,23:01:51,"73,88",°C,NaN,NaN
7,MOTEUR_BROYEUR,87812,2026-05-28,14:13:31,"80,66",°C,NaN,NaN
8,MOTEUR_BROYEUR,87812,2026-05-27,00:00:47,"85,49",°C,NaN,NaN
9,MOTEUR_BROYEUR,87812,2026-05-26,14:34:06,"88,36",°C,NaN,NaN


### 1.3 — Fix French Decimal Format

Several columns use the French format with commas as decimal separators (e.g., `80,37` instead of `80.37`). We need to convert these to proper floats.

In [3]:
def french_to_float(value):
    """Convert French decimal format (comma) to float."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    # Replace comma with dot and convert
    return float(str(value).replace(',', '.'))


# Rename columns to cleaner names
df = df_raw.copy()
df.columns = [
    'equipment',         # Poste Technique
    'sensor_id',         # Point de mesure
    'date',              # Date
    'time',              # Heure de la mesure
    'value',             # ValMes./Val.tot.cpteur
    'unit',              # Unité caractérist.
    'lower_limit',       # LimiteInférPlageMesure
    'upper_limit'        # LimiteSuperPlageMesure
]

# Convert French decimal columns to float
for col in ['value', 'lower_limit', 'upper_limit']:
    df[col] = df[col].apply(french_to_float)

print('Decimal conversion done ✓')
print(f'\nConverted dtypes:\n{df[["value", "lower_limit", "upper_limit"]].dtypes}')
print(f'\nSample values (first 5):\n{df[["equipment", "sensor_id", "value", "unit", "lower_limit", "upper_limit"]].head()}')

Decimal conversion done ✓

Converted dtypes:
value          float64
lower_limit    float64
upper_limit    float64
dtype: object

Sample values (first 5):
        equipment  sensor_id  value unit  lower_limit  upper_limit
0  MOTEUR_BROYEUR      87812  80.37   °C          NaN          NaN
1  MOTEUR_BROYEUR      87812  28.03   °C          NaN          NaN
2  MOTEUR_BROYEUR      87812  55.90   °C          NaN          NaN
3  MOTEUR_BROYEUR      87812  80.29   °C          NaN          NaN
4  MOTEUR_BROYEUR      87812  51.50   °C          NaN          NaN


### 1.4 — Create Proper DateTime Column

Combine the `date` and `time` columns into a single `datetime` column and sort chronologically.

In [4]:
# Combine date and time into a single datetime column
df['datetime'] = pd.to_datetime(
    df['date'].astype(str).str[:10] + ' ' + df['time'].astype(str),
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

# Sort by equipment, sensor, and datetime
df = df.sort_values(['equipment', 'sensor_id', 'datetime']).reset_index(drop=True)

# Drop the separate date/time columns (now redundant)
df = df.drop(columns=['date', 'time'])

print(f'DateTime range: {df["datetime"].min()} → {df["datetime"].max()}')
print(f'Total span: {(df["datetime"].max() - df["datetime"].min()).days} days')
print(f'\nDatetime null count: {df["datetime"].isna().sum()}')
df[['equipment', 'sensor_id', 'datetime', 'value', 'unit']].head(10)

DateTime range: 2026-05-05 00:00:31 → 2026-06-05 12:57:06
Total span: 31 days

Datetime null count: 0


,equipment,sensor_id,datetime,value,unit
0,BROYEUR_A_BOULETS,132813,2026-05-05 00:49:02,0.03,mm/s
1,BROYEUR_A_BOULETS,132813,2026-05-06 13:39:42,0.13,mm/s
2,BROYEUR_A_BOULETS,132813,2026-05-07 00:00:34,0.03,mm/s
3,BROYEUR_A_BOULETS,132813,2026-05-08 00:00:35,0.03,mm/s
4,BROYEUR_A_BOULETS,132813,2026-05-09 00:00:36,0.03,mm/s
5,BROYEUR_A_BOULETS,132813,2026-05-10 00:00:05,0.03,mm/s
6,BROYEUR_A_BOULETS,132813,2026-05-11 10:14:10,0.43,mm/s
7,BROYEUR_A_BOULETS,132813,2026-05-12 00:02:06,0.52,mm/s
8,BROYEUR_A_BOULETS,132813,2026-05-13 00:00:39,0.48,mm/s
9,BROYEUR_A_BOULETS,132813,2026-05-14 00:00:40,0.48,mm/s


### 1.5 — Handle Missing Values

In [5]:
print('=== Missing Values Before Cleaning ===')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df)}')

# Check the rows with missing 'value' or 'unit'
missing_value_rows = df[df['value'].isna() | df['unit'].isna()]
if len(missing_value_rows) > 0:
    print(f'\nRows with missing value or unit ({len(missing_value_rows)}):')
    print(missing_value_rows)

# Drop rows where the measurement value itself is missing (can't use these)
df = df.dropna(subset=['value']).copy()

# For lower_limit: fill with 0 where missing (assumes no lower bound)
# For upper_limit: leave as NaN — we'll handle per-sensor later
df['lower_limit'] = df['lower_limit'].fillna(0)

print(f'\n=== After Cleaning ===')
print(f'Rows remaining: {len(df)}')
print(f'\nMissing values:\n{df.isnull().sum()}')

=== Missing Values Before Cleaning ===
equipment        0
sensor_id        0
value            1
unit             1
lower_limit    550
upper_limit    135
datetime         0
dtype: int64

Total rows: 584

Rows with missing value or unit (1):
           equipment  sensor_id  value unit  lower_limit  upper_limit  \
523  TURBOSOUFFLANTE     125487    NaN  NaN          NaN          NaN   

               datetime  
523 2026-05-15 15:35:27  

=== After Cleaning ===
Rows remaining: 583

Missing values:
equipment        0
sensor_id        0
value            0
unit             0
lower_limit      0
upper_limit    134
datetime         0
dtype: int64


### 1.6 — Data Summary per Equipment & Sensor

In [6]:
# Summary table: one row per (equipment, sensor)
summary = df.groupby(['equipment', 'sensor_id', 'unit']).agg(
    count=('value', 'count'),
    mean=('value', 'mean'),
    std=('value', 'std'),
    min_val=('value', 'min'),
    max_val=('value', 'max'),
    upper_limit=('upper_limit', 'first'),
    date_min=('datetime', 'min'),
    date_max=('datetime', 'max')
).round(2)

# Add % readings over the upper limit
def pct_over_limit(group):
    if group['upper_limit'].isna().all():
        return np.nan
    limit = group['upper_limit'].dropna().iloc[0]
    return round((group['value'] > limit).mean() * 100, 1)

pct_over = df.groupby(['equipment', 'sensor_id', 'unit']).apply(pct_over_limit)
summary['pct_over_limit'] = pct_over.values

print('=== Complete Sensor Summary ===')
summary

=== Complete Sensor Summary ===


count     mean      std  min_val  max_val  \
equipment           sensor_id unit                                              
BROYEUR_A_BOULETS   132813    mm/s     31     0.40     0.18     0.03     0.52   
                    132815    mm/s     31     0.34     0.14     0.03     0.46   
                    132817    mm/s     31     0.61     0.28     0.03     0.75   
                    132819    mm/s     31     0.52     0.23     0.03     0.66   
                    132821    mm/s     31     0.64     0.28     0.05     0.79   
                    132823    mm/s     31     0.37     0.15     0.04     0.46   
                    132825    mm/s     31     0.50     0.20     0.07     0.81   
                    132827    mm/s     31     0.93     0.42     0.05     1.18   
MOTEUR_BROYEUR      87812     °C       30    79.00    12.70    28.03    88.96   
                    87813     °C       30    79.00    12.70    28.03    88.96   
                    87814     °C       30    79.00    12.70    28.03    88.96   
                    87815     °C       30    79.00    12.70    28.03    88.96   
MOTEUR_ENTRAINEMENT 138826    °C       31    31.08     6.55    18.89    44.29   
                    138828    mm/s     31     0.91     2.36     0.07     8.94   
                    138830    mm/s     31     0.37     0.94     0.05     3.52   
                    138832    mm/s     31     1.30     2.92     0.00    15.42   
                    138834    mm/s     31     0.12     0.24     0.00     1.01   
TURBOSOUFFLANTE     125498    mm        4   118.25     4.03   115.00   124.00   
                    125499    Bar       4 -7411.25  1309.21 -8624.00 -5644.80   
                    125500    Bar       4     5.72     1.30     3.80     6.50   
                    125501    Bar       3     0.50     0.00     0.50     0.50   
                    125504    Bar       3     2.80     0.40     2.40     3.20   
                    125505    °C        4    24.50     4.12    20.00    28.00   
                    125506    °C        4    33.00     6.68    25.00    39.00   
                    125507    °C        4    49.00     7.39    38.00    54.00   
                    266211    mm/s     30    23.20     7.41    11.92    34.49   

                                    upper_limit            date_min  \
equipment           sensor_id unit                                    
BROYEUR_A_BOULETS   132813    mm/s         11.0 2026-05-05 00:49:02   
                    132815    mm/s         11.0 2026-05-05 01:15:52   
                    132817    mm/s         11.0 2026-05-05 01:18:09   
                    132819    mm/s         11.0 2026-05-05 00:04:54   
                    132821    mm/s         11.0 2026-05-05 00:34:51   
                    132823    mm/s         11.0 2026-05-05 00:47:49   
                    132825    mm/s         11.0 2026-05-05 00:20:05   
                    132827    mm/s         11.0 2026-05-05 00:20:58   
MOTEUR_BROYEUR      87812     °C            NaN 2026-05-05 03:34:37   
                    87813     °C            NaN 2026-05-05 03:34:37   
                    87814     °C            NaN 2026-05-05 03:34:37   
                    87815     °C            NaN 2026-05-05 03:34:37   
MOTEUR_ENTRAINEMENT 138826    °C           90.0 2026-05-05 14:30:11   
                    138828    mm/s         11.0 2026-05-05 14:04:38   
                    138830    mm/s         11.0 2026-05-05 14:07:05   
                    138832    mm/s         11.0 2026-05-05 14:10:14   
                    138834    mm/s         11.0 2026-05-05 00:00:31   
TURBOSOUFFLANTE     125498    mm            NaN 2026-05-08 11:13:42   
                    125499    Bar       -6864.0 2026-05-08 11:13:42   
                    125500    Bar           NaN 2026-05-08 11:13:42   
                    125501    Bar           NaN 2026-05-08 11:13:42   
                    125504    Bar           NaN 2026-05-15 15:35:27   
                    125505    °C           90.0 2026-05-08 11:1

### 1.7 — Equipment Overview

In [7]:
print('=== Equipment Overview ===')
for eq in df['equipment'].unique():
    sub = df[df['equipment'] == eq]
    sensors = sub['sensor_id'].nunique()
    units = sub['unit'].dropna().unique()
    n = len(sub)
    date_range = f"{sub['datetime'].min().strftime('%Y-%m-%d')} → {sub['datetime'].max().strftime('%Y-%m-%d')}"
    
    print(f'\n╔══ {eq} ══╗')
    print(f'  Sensors: {sensors} | Readings: {n} | Date range: {date_range}')
    print(f'  Measurement types: {list(units)}')
    
    for sid in sorted(sub['sensor_id'].unique()):
        s = sub[sub['sensor_id'] == sid]
        v = s['value'].dropna()
        u = s['unit'].dropna().iloc[0] if len(s['unit'].dropna()) > 0 else '?'
        ulim = s['upper_limit'].dropna()
        lim_str = f' | Limit: {ulim.iloc[0]}' if len(ulim) > 0 else ' | Limit: N/A'
        print(f'    Sensor {sid} ({u}): {len(v)} pts, '
              f'range [{v.min():.2f} — {v.max():.2f}], '
              f'mean={v.mean():.2f}{lim_str}')

=== Equipment Overview ===

╔══ BROYEUR_A_BOULETS ══╗
  Sensors: 8 | Readings: 248 | Date range: 2026-05-05 → 2026-06-04
  Measurement types: ['mm/s']
    Sensor 132813 (mm/s): 31 pts, range [0.03 — 0.52], mean=0.40 | Limit: 11.0
    Sensor 132815 (mm/s): 31 pts, range [0.03 — 0.46], mean=0.34 | Limit: 11.0
    Sensor 132817 (mm/s): 31 pts, range [0.03 — 0.75], mean=0.61 | Limit: 11.0
    Sensor 132819 (mm/s): 31 pts, range [0.03 — 0.66], mean=0.52 | Limit: 11.0
    Sensor 132821 (mm/s): 31 pts, range [0.05 — 0.79], mean=0.64 | Limit: 11.0
    Sensor 132823 (mm/s): 31 pts, range [0.04 — 0.46], mean=0.37 | Limit: 11.0
    Sensor 132825 (mm/s): 31 pts, range [0.07 — 0.81], mean=0.50 | Limit: 11.0
    Sensor 132827 (mm/s): 31 pts, range [0.05 — 1.18], mean=0.93 | Limit: 11.0

╔══ MOTEUR_BROYEUR ══╗
  Sensors: 4 | Readings: 120 | Date range: 2026-05-05 → 2026-06-04
  Measurement types: ['°C']
    Sensor 87812 (°C): 30 pts, range [28.03 — 88.96], mean=79.00 | Limit: N/A
    Sensor 87813 (°C

### 1.8 — Save Cleaned Data

In [8]:
# Save the cleaned DataFrame for subsequent steps
df.to_csv('data_cleaned.csv', index=False)
print(f'Cleaned data saved to data_cleaned.csv ✓')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nFinal dtypes:\n{df.dtypes}')
df.head()

Cleaned data saved to data_cleaned.csv ✓
Shape: (583, 7)
Columns: ['equipment', 'sensor_id', 'value', 'unit', 'lower_limit', 'upper_limit', 'datetime']

Final dtypes:
equipment                 str
sensor_id               int64
value                 float64
unit                      str
lower_limit           float64
upper_limit           float64
datetime       datetime64[us]
dtype: object


,equipment,sensor_id,value,unit,lower_limit,upper_limit,datetime
0,BROYEUR_A_BOULETS,132813,0.03,mm/s,0.0,11.0,2026-05-05 00:49:02
1,BROYEUR_A_BOULETS,132813,0.13,mm/s,0.0,11.0,2026-05-06 13:39:42
2,BROYEUR_A_BOULETS,132813,0.03,mm/s,0.0,11.0,2026-05-07 00:00:34
3,BROYEUR_A_BOULETS,132813,0.03,mm/s,0.0,11.0,2026-05-08 00:00:35
4,BROYEUR_A_BOULETS,132813,0.03,mm/s,0.0,11.0,2026-05-09 00:00:36


---

**✅ Step 1 Complete** — Data is cleaned, decimals fixed, datetime created, missing values handled.

**Next: Step 2 — Exploratory Data Analysis (EDA)**